# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [ ]:
pip install ipywidgets numpy matplotlib scipy

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
from scipy.ndimage import gaussian_filter1d
import ipywidgets as widgets
from IPython.display import display

def generate_signal(bit_sequence, samples_per_bit, pulse_shape='nrz'):
    """Генерация сигнала из битовой последовательности"""
    signal = np.repeat(bit_sequence, samples_per_bit)

    # Применяем форму импульса
    if pulse_shape == 'nrz':
        # NRZ - без возврата к нулю
        pass  # Уже сгенерирован
    elif pulse_shape == 'rz':
        # RZ - с возвратом к нулю
        for i in range(len(bit_sequence)):
            start_idx = i * samples_per_bit
            mid_idx = start_idx + samples_per_bit // 2
            signal[mid_idx:start_idx + samples_per_bit] = 0

    return signal

def apply_channel_effects(signal, samples_per_bit, dispersion=0.1, attenuation=0.1, noise_level=0.05, nonlinearity=0):
    """Применение эффектов канала передачи"""
    distorted_signal = signal.copy()

    # 1. Затухание
    distorted_signal *= (1 - attenuation)

    # 2. Дисперсия (моделируем ФНЧ)
    if dispersion > 0:
        cutoff_freq = max(0.01, 0.5 - dispersion * 0.4)  # Чем больше дисперсия, тем уже полоса
        b, a = sp_signal.butter(3, cutoff_freq)
        distorted_signal = sp_signal.lfilter(b, a, distorted_signal)

    # 3. Нелинейные эффекты (простая модель)
    if nonlinearity > 0:
        distorted_signal = np.tanh(distorted_signal * (1 + nonlinearity))

    # 4. Шум
    if noise_level > 0:
        noise = np.random.normal(0, noise_level, len(distorted_signal))
        distorted_signal += noise

    return distorted_signal

def plot_eye_diagram(bit_rate=10e9, sequence_length=500, samples_per_bit=32,
                    dispersion=0.3, attenuation=0.2, noise_level=0.08,
                    nonlinearity=0.1, jitter=0.05, show_ideal=False):
    """Построение интерактивной глаз-диаграммы"""

    # Параметры времени
    bit_period = 1 / bit_rate
    t_bit = np.linspace(0, bit_period, samples_per_bit, endpoint=False)
    t_eye = np.linspace(0, 2 * bit_period, 2 * samples_per_bit)  # 2 бита для глаз-диаграммы

    # Генерация PRBS
    bits = np.random.randint(0, 2, sequence_length)
    ideal_signal = generate_signal(bits, samples_per_bit, 'nrz')

    # Применение эффектов канала
    distorted_signal = apply_channel_effects(ideal_signal, samples_per_bit,
                                           dispersion, attenuation, noise_level, nonlinearity)

    # Добавление джиттера
    if jitter > 0:
        jitter_samples = int(jitter * samples_per_bit)
        jittered_signal = np.zeros_like(distorted_signal)
        for i in range(sequence_length):
            start_idx = i * samples_per_bit
            end_idx = start_idx + samples_per_bit
            jitter_offset = np.random.randint(-jitter_samples, jitter_samples + 1)
            jittered_start = max(0, start_idx + jitter_offset)
            jittered_end = min(len(distorted_signal), end_idx + jitter_offset)
            orig_start = max(0, start_idx - jitter_offset)
            orig_end = min(len(distorted_signal), end_idx - jitter_offset)

            actual_length = min(jittered_end - jittered_start, orig_end - orig_start)
            if actual_length > 0:
                jittered_signal[jittered_start:jittered_start + actual_length] = \
                    distorted_signal[orig_start:orig_start + actual_length]
        distorted_signal = jittered_signal

    # Построение глаз-диаграммы
    eye_segments = []
    ideal_segments = []

    # Начинаем с некоторого смещения чтобы избежать переходных процессов
    start_offset = 2 * samples_per_bit

    for i in range(start_offset, len(distorted_signal) - 2 * samples_per_bit, samples_per_bit):
        # Для искаженного сигнала
        segment = distorted_signal[i: i + 2 * samples_per_bit]
        if len(segment) == 2 * samples_per_bit:
            eye_segments.append(segment)

        # Для идеального сигнала (если нужно показать)
        if show_ideal:
            ideal_segment = ideal_signal[i: i + 2 * samples_per_bit]
            if len(ideal_segment) == 2 * samples_per_bit:
                ideal_segments.append(ideal_segment)

    # Создание графиков
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # График 1: Глаз-диаграмма
    for segment in eye_segments[:200]:  # Ограничиваем количество сегментов для производительности
        ax1.plot(t_eye * 1e9, segment, 'b-', alpha=0.1, linewidth=0.8)

    ax1.set_title('Глаз-диаграмма сигнала')
    ax1.set_xlabel('Время, нс')
    ax1.set_ylabel('Амплитуда, отн. ед.')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([-0.5, 1.5])

    # График 2: Сравнение идеального и искаженного сигнала
    time_span = min(10 * samples_per_bit, len(ideal_signal))
    t_plot = np.arange(time_span) * bit_period * 1e9

    ax2.plot(t_plot, ideal_signal[:time_span], 'g-', label='Идеальный сигнал', linewidth=2, alpha=0.7)
    ax2.plot(t_plot, distorted_signal[:time_span], 'r-', label='Искаженный сигнал', alpha=0.8)
    ax2.set_title('Сравнение сигналов')
    ax2.set_xlabel('Время, нс')
    ax2.set_ylabel('Амплитуда, отн. ед.')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Расчет и вывод параметров качества
    if len(eye_segments) > 0:
        eye_array = np.array(eye_segments)
        eye_height = np.max(eye_array[:, samples_per_bit//2]) - np.min(eye_array[:, samples_per_bit//2])
        eye_width_metric = samples_per_bit * 0.8  # Упрощенная метрика

        print(f"Качество сигнала:")
        print(f"  - Высота глаза: {eye_height:.3f}")
        print(f"  - Ширина глаза: {eye_width_metric:.1f} samples")
        print(f"  - Открытие глаза: {(eye_height/eye_width_metric)*100:.1f}%")

        # Оценка BER (очень упрощенная)
        noise_std = np.std(eye_array[:, samples_per_bit//2])
        if noise_std > 0:
            q_factor = eye_height / (2 * noise_std)
            ber_estimate = 0.5 * sp_signal.erfc(q_factor / np.sqrt(2))
            print(f"  - Оценка BER: {ber_estimate:.2e}")

# Создание интерактивных ползунков
def create_interactive_eye_diagram():
    """Создание интерактивного интерфейса с ползунками"""

    style = {'description_width': 'initial'}

    # Параметры для ползунков
    bit_rate_slider = widgets.FloatLogSlider(
        value=10e9,
        base=10,
        min=9, max=11,
        step=0.1,
        description='Скорость (бит/с):',
        readout_format='.0e',
        style=style
    )

    dispersion_slider = widgets.FloatSlider(
        value=0.3,
        min=0.0,
        max=1.0,
        step=0.05,
        description='Дисперсия:',
        style=style
    )

    attenuation_slider = widgets.FloatSlider(
        value=0.2,
        min=0.0,
        max=0.8,
        step=0.05,
        description='Затухание:',
        style=style
    )

    noise_slider = widgets.FloatSlider(
        value=0.08,
        min=0.0,
        max=0.3,
        step=0.01,
        description='Уровень шума:',
        style=style
    )

    nonlinearity_slider = widgets.FloatSlider(
        value=0.1,
        min=0.0,
        max=0.5,
        step=0.05,
        description='Нелинейность:',
        style=style
    )

    jitter_slider = widgets.FloatSlider(
        value=0.05,
        min=0.0,
        max=0.2,
        step=0.01,
        description='Джиттер:',
        style=style
    )

    samples_slider = widgets.IntSlider(
        value=32,
        min=16,
        max=64,
        step=8,
        description='Сэмплов на бит:',
        style=style
    )

    sequence_slider = widgets.IntSlider(
        value=500,
        min=100,
        max=1000,
        step=100,
        description='Длина последовательности:',
        style=style
    )

    ideal_checkbox = widgets.Checkbox(
        value=False,
        description='Показать идеальный сигнал',
        style=style
    )

    # Объединение всех элементов управления
    controls = widgets.interactive(plot_eye_diagram,
        bit_rate=bit_rate_slider,
        sequence_length=sequence_slider,
        samples_per_bit=samples_slider,
        dispersion=dispersion_slider,
        attenuation=attenuation_slider,
        noise_level=noise_slider,
        nonlinearity=nonlinearity_slider,
        jitter=jitter_slider,
        show_ideal=ideal_checkbox
    )

    # Отображение интерфейса
    display(controls)

# Запуск интерактивного интерфейса
print("Интерактивная модель глаз-диаграммы для ВОЛС")
print("Используйте ползунки для изменения параметров канала передачи")
create_interactive_eye_diagram()

Интерактивная модель глаз-диаграммы для ВОЛС
Используйте ползунки для изменения параметров канала передачи


interactive(children=(FloatLogSlider(value=10000000000.0, description='Скорость (бит/с):', max=11.0, min=9.0, …

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
from scipy.ndimage import gaussian_filter1d
import ipywidgets as widgets
from IPython.display import display

def generate_signal(bit_sequence, samples_per_bit, pulse_shape='nrz'):
    """Генерация сигнала из битовой последовательности"""
    signal = np.repeat(bit_sequence, samples_per_bit)

    # Применяем форму импульса
    if pulse_shape == 'nrz':
        # NRZ - без возврата к нулю
        pass  # Уже сгенерирован
    elif pulse_shape == 'rz':
        # RZ - с возвратом к нулю
        for i in range(len(bit_sequence)):
            start_idx = i * samples_per_bit
            mid_idx = start_idx + samples_per_bit // 2
            signal[mid_idx:start_idx + samples_per_bit] = 0

    return signal

def apply_channel_effects(signal, samples_per_bit, dispersion=0.1, attenuation=0.1, noise_level=0.05, nonlinearity=0):
    """Применение эффектов канала передачи"""
    distorted_signal = signal.copy()

    # 1. Затухание
    distorted_signal *= (1 - attenuation)

    # 2. Дисперсия (моделируем ФНЧ)
    if dispersion > 0:
        cutoff_freq = max(0.01, 0.5 - dispersion * 0.4)  # Чем больше дисперсия, тем уже полоса
        b, a = sp_signal.butter(3, cutoff_freq)
        distorted_signal = sp_signal.lfilter(b, a, distorted_signal)

    # 3. Нелинейные эффекты (простая модель)
    if nonlinearity > 0:
        distorted_signal = np.tanh(distorted_signal * (1 + nonlinearity))

    # 4. Шум
    if noise_level > 0:
        noise = np.random.normal(0, noise_level, len(distorted_signal))
        distorted_signal += noise

    return distorted_signal

def plot_eye_diagram(bit_rate=10e9, sequence_length=500, samples_per_bit=32,
                    dispersion=0.3, attenuation=0.2, noise_level=0.08,
                    nonlinearity=0.1, jitter=0.05, show_ideal=False):
    """Построение интерактивной глаз-диаграммы"""

    # Параметры времени
    bit_period = 1 / bit_rate
    t_bit = np.linspace(0, bit_period, samples_per_bit, endpoint=False)
    t_eye = np.linspace(0, 2 * bit_period, 2 * samples_per_bit)  # 2 бита для глаз-диаграммы

    # Генерация PRBS
    bits = np.random.randint(0, 2, sequence_length)
    ideal_signal = generate_signal(bits, samples_per_bit, 'nrz')

    # Применение эффектов канала
    distorted_signal = apply_channel_effects(ideal_signal, samples_per_bit,
                                           dispersion, attenuation, noise_level, nonlinearity)

    # Добавление джиттера
    if jitter > 0:
        jitter_samples = int(jitter * samples_per_bit)
        jittered_signal = np.zeros_like(distorted_signal)
        for i in range(sequence_length):
            start_idx = i * samples_per_bit
            end_idx = start_idx + samples_per_bit
            jitter_offset = np.random.randint(-jitter_samples, jitter_samples + 1)
            jittered_start = max(0, start_idx + jitter_offset)
            jittered_end = min(len(distorted_signal), end_idx + jitter_offset)
            orig_start = max(0, start_idx - jitter_offset)
            orig_end = min(len(distorted_signal), end_idx - jitter_offset)

            actual_length = min(jittered_end - jittered_start, orig_end - orig_start)
            if actual_length > 0:
                jittered_signal[jittered_start:jittered_start + actual_length] = \
                    distorted_signal[orig_start:orig_start + actual_length]
        distorted_signal = jittered_signal

    # Построение глаз-диаграммы
    eye_segments = []
    ideal_segments = []

    # Начинаем с некоторого смещения чтобы избежать переходных процессов
    start_offset = 2 * samples_per_bit

    for i in range(start_offset, len(distorted_signal) - 2 * samples_per_bit, samples_per_bit):
        # Для искаженного сигнала
        segment = distorted_signal[i: i + 2 * samples_per_bit]
        if len(segment) == 2 * samples_per_bit:
            eye_segments.append(segment)

        # Для идеального сигнала (если нужно показать)
        if show_ideal:
            ideal_segment = ideal_signal[i: i + 2 * samples_per_bit]
            if len(ideal_segment) == 2 * samples_per_bit:
                ideal_segments.append(ideal_segment)

    # Создание графиков
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # График 1: Глаз-диаграмма
    for segment in eye_segments[:200]:  # Ограничиваем количество сегментов для производительности
        ax1.plot(t_eye * 1e9, segment, 'b-', alpha=0.1, linewidth=0.8)

    ax1.set_title('Глаз-диаграмма сигнала')
    ax1.set_xlabel('Время, нс')
    ax1.set_ylabel('Амплитуда, отн. ед.')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([-0.5, 1.5])

    # График 2: Сравнение идеального и искаженного сигнала
    time_span = min(10 * samples_per_bit, len(ideal_signal))
    t_plot = np.arange(time_span) * bit_period * 1e9

    ax2.plot(t_plot, ideal_signal[:time_span], 'g-', label='Идеальный сигнал', linewidth=2, alpha=0.7)
    ax2.plot(t_plot, distorted_signal[:time_span], 'r-', label='Искаженный сигнал', alpha=0.8)
    ax2.set_title('Сравнение сигналов')
    ax2.set_xlabel('Время, нс')
    ax2.set_ylabel('Амплитуда, отн. ед.')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Расчет и вывод параметров качества
    if len(eye_segments) > 0:
        eye_array = np.array(eye_segments)
        eye_height = np.max(eye_array[:, samples_per_bit//2]) - np.min(eye_array[:, samples_per_bit//2])
        eye_width_metric = samples_per_bit * 0.8  # Упрощенная метрика

        print(f"Качество сигнала:")
        print(f"  - Высота глаза: {eye_height:.3f}")
        print(f"  - Ширина глаза: {eye_width_metric:.1f} samples")
        print(f"  - Открытие глаза: {(eye_height/eye_width_metric)*100:.1f}%")

        # Оценка BER (очень упрощенная)
        noise_std = np.std(eye_array[:, samples_per_bit//2])
        if noise_std > 0:
            q_factor = eye_height / (2 * noise_std)
            ber_estimate = 0.5 * sp_signal.erfc(q_factor / np.sqrt(2))
            print(f"  - Оценка BER: {ber_estimate:.2e}")

# Создание интерактивных ползунков
def create_interactive_eye_diagram():
    """Создание интерактивного интерфейса с ползунками"""

    style = {'description_width': 'initial'}

    # Параметры для ползунков
    bit_rate_slider = widgets.FloatLogSlider(
        value=10e9,
        base=10,
        min=9, max=11,
        step=0.1,
        description='Скорость (бит/с):',
        readout_format='.0e',
        style=style
    )

    dispersion_slider = widgets.FloatSlider(
        value=0.3,
        min=0.0,
        max=1.0,
        step=0.05,
        description='Дисперсия:',
        style=style
    )

    attenuation_slider = widgets.FloatSlider(
        value=0.2,
        min=0.0,
        max=0.8,
        step=0.05,
        description='Затухание:',
        style=style
    )

    noise_slider = widgets.FloatSlider(
        value=0.08,
        min=0.0,
        max=0.3,
        step=0.01,
        description='Уровень шума:',
        style=style
    )

    nonlinearity_slider = widgets.FloatSlider(
        value=0.1,
        min=0.0,
        max=0.5,
        step=0.05,
        description='Нелинейность:',
        style=style
    )

    jitter_slider = widgets.FloatSlider(
        value=0.05,
        min=0.0,
        max=0.2,
        step=0.01,
        description='Джиттер:',
        style=style
    )

    samples_slider = widgets.IntSlider(
        value=32,
        min=16,
        max=64,
        step=8,
        description='Сэмплов на бит:',
        style=style
    )

    sequence_slider = widgets.IntSlider(
        value=500,
        min=100,
        max=1000,
        step=100,
        description='Длина последовательности:',
        style=style
    )

    ideal_checkbox = widgets.Checkbox(
        value=False,
        description='Показать идеальный сигнал',
        style=style
    )

    # Объединение всех элементов управления
    controls = widgets.interactive(plot_eye_diagram,
        bit_rate=bit_rate_slider,
        sequence_length=sequence_slider,
        samples_per_bit=samples_slider,
        dispersion=dispersion_slider,
        attenuation=attenuation_slider,
        noise_level=noise_slider,
        nonlinearity=nonlinearity_slider,
        jitter=jitter_slider,
        show_ideal=ideal_checkbox
    )

    # Отображение интерфейса
    display(controls)

# Запуск интерактивного интерфейса
print("Интерактивная модель глаз-диаграммы для ВОЛС")
print("Используйте ползунки для изменения параметров канала передачи")
create_interactive_eye_diagram()

Интерактивная модель глаз-диаграммы для ВОЛС
Используйте ползунки для изменения параметров канала передачи


interactive(children=(FloatLogSlider(value=10000000000.0, description='Скорость (бит/с):', max=11.0, min=9.0, …

In [4]:
# Вставьте весь этот код в одну ячейку Google Colab

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
from ipywidgets import interact, widgets
import warnings
warnings.filterwarnings('ignore')

def generate_signal(bit_sequence, samples_per_bit):
    """Генерация сигнала из битовой последовательности"""
    return np.repeat(bit_sequence.astype(float), samples_per_bit)

def apply_channel_effects(signal, dispersion=0.1, attenuation=0.1, noise_level=0.05):
    """Применение эффектов канала передачи"""
    distorted = signal.copy().astype(float)

    # Затухание
    distorted *= (1.0 - attenuation)

    # Дисперсия
    if dispersion > 0:
        cutoff_freq = max(0.01, 0.5 - dispersion * 0.4)
        b, a = sp_signal.butter(3, cutoff_freq)
        distorted = sp_signal.lfilter(b, a, distorted)

    # Шум
    if noise_level > 0:
        noise = np.random.normal(0.0, noise_level, len(distorted))
        distorted += noise

    return distorted

def create_eye_diagram(bit_rate=10e9, sequence_length=500, samples_per_bit=32,
                       dispersion=0.3, attenuation=0.2, noise_level=0.08):

    # Параметры времени
    bit_period = 1.0 / bit_rate

    # Генерация PRBS
    bits = np.random.randint(0, 2, sequence_length)
    ideal_signal = generate_signal(bits, samples_per_bit)

    # Применение эффектов канала
    distorted_signal = apply_channel_effects(ideal_signal, dispersion,
                                           attenuation, noise_level)

    # Построение глаз-диаграммы
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Левый график: глаз-диаграмма
    eye_segments = []
    for i in range(2*samples_per_bit, len(distorted_signal) - 2*samples_per_bit, samples_per_bit):
        segment = distorted_signal[i:i + 2*samples_per_bit]
        if len(segment) == 2*samples_per_bit:
            eye_segments.append(segment)

    t_eye = np.linspace(0, 2*bit_period*1e9, 2*samples_per_bit)
    for segment in eye_segments[:100]:
        ax1.plot(t_eye, segment, 'b-', alpha=0.1, linewidth=0.8)

    ax1.set_title('Глаз-диаграмма сигнала ВОЛС')
    ax1.set_xlabel('Время, нс')
    ax1.set_ylabel('Амплитуда')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([-0.5, 1.5])

    # Правый график: форма сигнала
    time_span = min(10*samples_per_bit, len(ideal_signal))
    t_plot = np.arange(time_span) * bit_period * 1e9

    ax2.plot(t_plot, ideal_signal[:time_span], 'g-', label='Идеальный', linewidth=2, alpha=0.7)
    ax2.plot(t_plot, distorted_signal[:time_span], 'r-', label='Искаженный', alpha=0.8)
    ax2.set_title('Форма сигнала')
    ax2.set_xlabel('Время, нс')
    ax2.set_ylabel('Амплитуда')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([-0.5, 1.5])

    plt.tight_layout()
    plt.show()

    # Оценка качества
    if eye_segments:
        eye_array = np.array(eye_segments)
        center = samples_per_bit
        eye_height = np.percentile(eye_array[:, center], 75) - np.percentile(eye_array[:, center], 25)

        print("="*50)
        print("ПАРАМЕТРЫ КАЧЕСТВА:")
        print("="*50)
        print(f"Высота глаза: {eye_height:.3f}")
        print(f"Ширина глаза: {2*bit_period*1e9*0.7:.1f} нс")
        print(f"Открытие глаза: {(eye_height)*100:.1f}%")

        if eye_height > 0.8:
            print("Качество: ОТЛИЧНОЕ ✓")
        elif eye_height > 0.5:
            print("Качество: ХОРОШЕЕ ✓")
        elif eye_height > 0.3:
            print("Качество: УДОВЛЕТВОРИТЕЛЬНОЕ ⚠")
        else:
            print("Качество: ПЛОХОЕ ✗")
        print("="*50)

# Создаем интерактивный интерфейс
print("ЛАБОРАТОРНАЯ РАБОТА: АНАЛИЗ ВОЛОКОННО-ОПТИЧЕСКИХ ЛИНИЙ СВЯЗИ")
print("Используйте ползунки для изменения параметров:")

# Используем interact для создания ползунков
interact(create_eye_diagram,
         bit_rate=widgets.FloatLogSlider(value=10e9, base=10, min=9, max=11, step=0.1,
                                         description='Скорость (Гбит/с):', readout_format='.1f'),
         sequence_length=widgets.IntSlider(value=500, min=100, max=1000, step=100,
                                          description='Длина последоват.:'),
         samples_per_bit=widgets.IntSlider(value=32, min=16, max=64, step=8,
                                          description='Сэмплов/бит:'),
         dispersion=widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05,
                                       description='Дисперсия:'),
         attenuation=widgets.FloatSlider(value=0.2, min=0.0, max=0.8, step=0.05,
                                        description='Затухание:'),
         noise_level=widgets.FloatSlider(value=0.08, min=0.0, max=0.3, step=0.01,
                                        description='Уровень шума:'));

ЛАБОРАТОРНАЯ РАБОТА: АНАЛИЗ ВОЛОКОННО-ОПТИЧЕСКИХ ЛИНИЙ СВЯЗИ
Используйте ползунки для изменения параметров:


interactive(children=(FloatLogSlider(value=10000000000.0, description='Скорость (Гбит/с):', max=11.0, min=9.0,…

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
import warnings
warnings.filterwarnings('ignore')

def generate_signal(bit_sequence, samples_per_bit):
    """Генерация сигнала из битовой последовательности"""
    return np.repeat(bit_sequence.astype(float), samples_per_bit)

def apply_channel_effects(signal, dispersion=0.1, attenuation=0.1,
                         noise_level=0.05, nonlinearity=0):
    """Применение эффектов канала передачи"""
    distorted = signal.copy().astype(float)

    # Затухание
    if attenuation > 0:
        distorted *= (1.0 - attenuation)

    # Дисперсия
    if dispersion > 0:
        cutoff_freq = max(0.01, 0.5 - dispersion * 0.4)
        b, a = sp_signal.butter(3, cutoff_freq)
        distorted = sp_signal.lfilter(b, a, distorted)

    # Нелинейные эффекты
    if nonlinearity > 0:
        distorted = np.tanh(distorted * (1.0 + nonlinearity))

    # Шум
    if noise_level > 0:
        noise = np.random.normal(0.0, noise_level, len(distorted))
        distorted += noise

    return distorted

# Простая версия без сложных виджетов
def simple_lab_work():
    """Упрощенная версия для EXE"""
    import tkinter as tk
    from tkinter import ttk
    from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

    def update_plot():
        # Получаем значения
        bit_rate = float(bit_rate_var.get()) * 1e9
        dispersion = float(dispersion_var.get())
        attenuation = float(attenuation_var.get())
        noise = float(noise_var.get())

        # Генерация сигнала
        sequence_length = 500
        samples_per_bit = 32
        bit_period = 1.0 / bit_rate

        bits = np.random.randint(0, 2, sequence_length)
        ideal_signal = generate_signal(bits, samples_per_bit)
        distorted_signal = apply_channel_effects(ideal_signal, dispersion,
                                               attenuation, noise)

        # Очистка и перерисовка
        plt.clf()

        # Создаем графики
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))

        # Глаз-диаграмма
        t_eye = np.linspace(0, 2*bit_period*1e9, 2*samples_per_bit)
        eye_segments = []

        for i in range(64, len(distorted_signal) - 2*samples_per_bit, samples_per_bit):
            segment = distorted_signal[i:i + 2*samples_per_bit]
            if len(segment) == 2*samples_per_bit:
                eye_segments.append(segment)

        for segment in eye_segments[:100]:
            ax1.plot(t_eye, segment, 'b-', alpha=0.1, linewidth=0.5)

        ax1.set_title(f'Глаз-диаграмма ВОЛС ({bit_rate/1e9} Гбит/с)')
        ax1.set_xlabel('Время, нс')
        ax1.set_ylabel('Амплитуда')
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim([-0.5, 1.5])

        # Форма сигнала
        time_span = min(10*samples_per_bit, len(ideal_signal))
        t_plot = np.arange(time_span) * bit_period * 1e9

        ax2.plot(t_plot, ideal_signal[:time_span], 'g-', label='Идеальный',
                linewidth=1, alpha=0.7)
        ax2.plot(t_plot, distorted_signal[:time_span], 'r-', label='Искаженный',
                alpha=0.6)
        ax2.set_title('Форма сигнала')
        ax2.set_xlabel('Время, нс')
        ax2.set_ylabel('Амплитуда')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        ax2.set_ylim([-0.5, 1.5])

        plt.tight_layout()

        # Встраиваем в Tkinter
        for widget in plot_frame.winfo_children():
            widget.destroy()

        canvas = FigureCanvasTkAgg(fig, master=plot_frame)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    # Создание GUI
    root = tk.Tk()
    root.title("Лабораторная работа: Анализ ВОЛС")
    root.geometry("1200x800")

    # Панель управления
    control_frame = tk.Frame(root, bg='#f0f0f0')
    control_frame.pack(side=tk.LEFT, fill=tk.Y, padx=10, pady=10)

    tk.Label(control_frame, text="ПАРАМЕТРЫ ВОЛС",
            font=('Arial', 14, 'bold'), bg='#f0f0f0').pack(pady=10)

    # Переменные
    bit_rate_var = tk.StringVar(value="10")
    dispersion_var = tk.StringVar(value="0.3")
    attenuation_var = tk.StringVar(value="0.2")
    noise_var = tk.StringVar(value="0.08")

    # Создаем ползунки
    def create_slider(frame, label, var, min_val, max_val):
        tk.Label(frame, text=label, bg='#f0f0f0').pack(anchor='w')
        scale = ttk.Scale(frame, from_=min_val, to=max_val,
                         orient=tk.HORIZONTAL, length=200,
                         command=lambda v: var.set(f"{float(v):.2f}"))
        scale.set(float(var.get()))
        scale.pack(pady=5)

        value_label = tk.Label(frame, textvariable=var, bg='#f0f0f0',
                              width=10, relief='sunken')
        value_label.pack()
        return scale

    # Ползунки
    create_slider(control_frame, "Скорость (Гбит/с):", bit_rate_var, 1, 100)
    create_slider(control_frame, "Дисперсия:", dispersion_var, 0, 1)
    create_slider(control_frame, "Затухание:", attenuation_var, 0, 0.8)
    create_slider(control_frame, "Уровень шума:", noise_var, 0, 0.3)

    # Кнопки
    tk.Button(control_frame, text="Обновить график",
             command=update_plot, bg='#4CAF50', fg='white',
             font=('Arial', 10)).pack(pady=20)

    tk.Button(control_frame, text="Выйти",
             command=root.quit, bg='#f44336', fg='white',
             font=('Arial', 10)).pack()

    # Область для графика
    plot_frame = tk.Frame(root)
    plot_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True,
                   padx=10, pady=10)

    # Первоначальный график
    update_plot()

    root.mainloop()

if __name__ == "__main__":
    simple_lab_work()